# KA SS 26 · Challenge 2 — **Stage 8: non-SSL ConvNeXt-UNet pair (Colab)**

Self-contained Colab notebook. **The only thing it needs is `/content/kaggle.json`.**
Designed to be started and left alone.

### What this is and why

Stage 7's three members were siblings: same SSL encoder, same targets, same schedule. Their
ensemble reached local **0.8035** against a best member of **0.7999** — **+0.0036** for two extra
70-epoch runs. Averaging near-identical models smooths nothing that matters.

Stage 8 trains the *other* family: **plain ImageNet-21k ConvNeXt-Tiny, no SimSiam adaptation.**
That is the diversity axis the Stage 7 ensemble never had, and it is also a reconstruction of the
lost Stage 5 model (local 0.8279, public 0.5293).

Second axis: **`SEED_EROSION_FRAC`.** `t_seed = 0.65` won in Stage 5 with a grid topping out at
0.65 *and* won again in Stage 7 with 0.90 above it — it is a confirmed interior optimum, so the
seed *threshold* is not the binding constraint. The seed *target* is. Test predictions still run
107.8/image against a GT mean of 125.2, so `n2` shrinks the seed cores (0.45 → 0.58) to separate
touching cells more aggressively. One member is the control, one is the probe, and the pair
doubles as the ensemble.

### Members

| | encoder init | `SEED_EROSION_FRAC` | seed | crop |
|---|---|---|---|---|
| `n1` | ImageNet-21k (plain) | 0.45 (Stage 5 control) | 2026 | 448×576 |
| `n2` | ImageNet-21k (plain) | 0.58 (under-segmentation probe) | 43 | 400×512 |

Everything else is the Stage 5 schedule: 70 epochs, LR 3e-4, encoder ×0.25, freeze 2, warmup 150,
42 steps/epoch, 4× flip × 3-scale TTA.

### Outputs

- `submission.csv` — **downloaded to your device** at the end (and mirrored into the dataset).
- A Kaggle dataset **`stage_8_nonssl`**, pushed *after every member*, containing the checkpoints,
  the cached val/test probability maps, and `val_ids.json`.

Stage 9 (on Kaggle, inside the Stage 7 notebook) attaches that dataset and blends these maps
against the m1/m2/m3 maps already sitting in `/kaggle/working/maps`.

### Split safety

The Stage 7 notebook was configured to reproduce Stage 5's split and silently produced a
different one (338/86 with 421 groups, where Stage 5 got 339/85 with 424). Cell 6 here asserts
**338 / 86 / 421** and stops in the first three minutes if the split drifts, because a drifted
split makes the Stage 9 blend weight untunable. It also writes `val_ids.json` so Stage 9 can
verify identity rather than trust a count.

## 0 · Before you start

1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `kaggle.json` to `/content` (the file browser on the left, not a subfolder).
3. Runtime → **Run all**, then leave it. Expect roughly **2h15m**.

Nothing else is required. No Drive, no manual downloads.

In [1]:
# ── 0 · Kaggle credentials, competition data, environment ─────────────────────
import os, sys, json, subprocess, shutil, time
from pathlib import Path

T_START = time.time()

KJSON = Path("/content/kaggle.json")
assert KJSON.exists(), "Upload kaggle.json to /content first (file browser on the left)."
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy2(KJSON, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
KAGGLE_USER = json.loads(KJSON.read_text())["username"]
print("kaggle user:", KAGGLE_USER)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle", "timm"], check=True)

COMP = "ka-ss-26-challenge-2"
DATA = Path("/content/data")
if not (DATA / "train.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["kaggle", "competitions", "download", "-c", COMP, "-p", str(DATA)],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    zips = list(DATA.glob("*.zip"))
    assert zips, ("Competition download failed. Check that kaggle.json is valid and that you have "
                  "accepted the competition rules on the website.")
    for z in zips:
        subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(DATA)], check=True)
        z.unlink()
print("data ready:", sorted(p.name for p in DATA.iterdir())[:12])

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print("gpu:", torch.cuda.get_device_properties(0).name)

kaggle user: ghostylicious

 0:00<00:06, 88.1MB/s]
  5%|▌         | 29.0M/533M [00:00<00:05, 93.7MB/s]
  8%|▊         | 41.0M/533M [00:00<00:05, 96.6MB/s]
 10%|▉         | 51.0M/533M [00:00<00:06, 79.5MB/s]
 12%|█▏        | 64.0M/533M [00:00<00:05, 93.1MB/s]
 15%|█▍        | 78.0M/533M [00:00<00:04, 107MB/s] 
 17%|█▋        | 89.0M/533M [00:00<00:04, 104MB/s]
 20%|█▉        | 105M/533M [00:01<00:03, 113MB/s] 
 23%|██▎       | 121M/533M [00:01<00:03, 127MB/s]
 25%|██▌       | 134M/533M [00:01<00:03, 125MB/s]
 28%|██▊       | 147M/533M [00:01<00:03, 126MB/s]
 30%|███       | 162M/533M [00:01<00:02, 134MB/s]
 33%|███▎      | 175M/533M [00:01<00:04, 89.8MB/s]
 36%|███▋      | 194M/533M [00:01<00:03, 112MB/s] 
 39%|███▉      | 207M/533M [00:02<00:03, 108MB/s]
 42%|████▏     | 223M/533M [00:02<00:02, 116MB/s]
 45%|████▌     | 241M/533M [00:02<00:02, 132MB/s]
 48%|████▊     | 255M/533M [00:02<00:02, 124MB/s]
 50%|█████     | 268M/533M [00:02<00:02, 120MB/s]
 53%|█████▎    | 283M/533M [00:02<0

In [2]:
# ── 1 · Imports and environment ───────────────────────────────────────────────
import gc, math, random, pickle, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import scipy.ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.segmentation import watershed
from sklearn.cluster import KMeans

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
import timm

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

def amp_autocast():
    return _autocast("cuda", dtype=torch.float16) if _AMP_NEW else _autocast()

def make_scaler():
    return _GradScaler("cuda") if _AMP_NEW else _GradScaler()

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

def elapsed():
    return time.time() - T_START

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {(sec % 3600) // 60:02d}m {sec % 60:02d}s"

print("torch", torch.__version__, "| timm", timm.__version__, "| opencv", HAVE_CV2)

torch 2.11.0+cu128 | timm 1.0.28 | opencv True


In [3]:
# ── 2 · Configuration ─────────────────────────────────────────────────────────
class CFG:
    RUN_TAG = "stage8_nonssl"

    SMOKE_TEST = False      # ~12 min rehearsal: 1 member, 3 epochs, tiny grids
    RESUME = True
    STRICT_SPLIT = True     # stop if the split does not reproduce Stage 7's exactly

    # ── wall-clock budgets ────────────────────────────────────────────────────
    TOTAL_BUDGET_SEC  = 3.6 * 3600
    MEMBER_BUDGET_SEC = 75 * 60
    FINAL_RESERVE_SEC = 35 * 60

    # ── geometry ──────────────────────────────────────────────────────────────
    IMG_H, IMG_W = 520, 704
    PAD_MULT = 32

    # ── split: byte-identical policy to Stage 7 ───────────────────────────────
    SPLIT_SEED = 2026
    GROUP_SPLIT = True
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0
    CLUSTER_K = 2
    EXPECT_TRAIN, EXPECT_VAL, EXPECT_GROUPS = 338, 86, 421

    # ── architecture (NO SSL encoder - that is the whole point) ───────────────
    ENCODER = "convnext_tiny.fb_in22k_ft_in1k"
    ENCODER_FALLBACKS = ("convnext_tiny", "resnet50", "resnet34")
    DECODER_CHANNELS = (256, 128, 64, 32)
    OUT_CHANNELS = 2

    # ── targets ───────────────────────────────────────────────────────────────
    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0

    # ── supervised schedule (Stage 5) ─────────────────────────────────────────
    SUP_EPOCHS = 70
    SUP_BATCH = 8
    SUP_LR = 3e-4
    SUP_ENCODER_LR_MULT = 0.25
    SUP_WD = 1e-4
    FREEZE_ENCODER_EPOCHS = 2
    WARMUP_STEPS = 150
    STEPS_PER_EPOCH = 42
    EVAL_START = 24
    EVAL_EVERY = 2
    GRAD_CLIP = 5.0

    # n1 is the Stage 5 control; n2 probes the seed target, which is the lever that
    # is left now that t_seed = 0.65 is a confirmed interior optimum.
    MEMBERS = [
        dict(name="n1", seed=2026, crop=(448, 576), erosion=0.45),
        dict(name="n2", seed=43,   crop=(400, 512), erosion=0.58),
    ]

    # ── augmentation ──────────────────────────────────────────────────────────
    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_SCALE = (0.85, 1.20)
    AUG_BRIGHT = 0.15
    AUG_CONTRAST = 0.15
    AUG_NOISE = 0.02

    # ── inference ─────────────────────────────────────────────────────────────
    TTA_FLIPS = True
    TTA_SCALES = (0.85, 1.0, 1.15)

    # ── decode sweep ──────────────────────────────────────────────────────────
    # score_thr was perfectly degenerate in Stage 5 and chose 0.0 on its own in
    # Stage 7. The axis is dead; the budget goes to t_seed and min_area instead.
    T_CELL_GRID = [0.40, 0.45, 0.50, 0.55, 0.60]
    T_SEED_GRID = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
    MIN_AREA_GRID = [30, 50, 70, 90, 110]
    SCORE_THR_GRID = [0.0]
    SWEEP_SUBSET = 45
    SWEEP_REFINE_TOP = 8
    ROBUST_TOL = 0.004

    DEFAULT_POST = dict(t_cell=0.50, t_seed=0.60, min_area=60, score_thr=0.0)
    SUB_INDEX_COL = "idx"

    # ── Kaggle dataset that carries the outputs back ──────────────────────────
    PUSH_DATASET = True
    DATASET_SLUG = "stage-8-nonssl"
    DATASET_TITLE = "stage_8_nonssl"


if CFG.SMOKE_TEST:
    CFG.SUP_EPOCHS = 3
    CFG.EVAL_START = 1
    CFG.EVAL_EVERY = 1
    CFG.MEMBERS = CFG.MEMBERS[:1]
    CFG.TTA_SCALES = (1.0,)
    CFG.T_CELL_GRID = [0.45, 0.50]
    CFG.T_SEED_GRID = [0.60, 0.65]
    CFG.MIN_AREA_GRID = [50, 90]
    CFG.SWEEP_SUBSET = 12
    CFG.MEMBER_BUDGET_SEC = 10 * 60
    CFG.FINAL_RESERVE_SEC = 12 * 60
    print(">>> SMOKE TEST - results are meaningless, this only proves the pipeline runs")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SPLIT_SEED)
STATS = {"run_tag": CFG.RUN_TAG, "smoke_test": CFG.SMOKE_TEST}
print("run tag:", CFG.RUN_TAG, "| members:", [m["name"] for m in CFG.MEMBERS])

run tag: stage8_nonssl | members: ['n1', 'n2']


In [5]:
# ── 3 · Paths and encoder preflight ───────────────────────────────────────────
WORK = Path("/content/work")
CKPT, ART, CACHE, MAPS, PUSH = (WORK / "ckpt", WORK / "artifacts",
                                WORK / "cache", WORK / "maps", Path("/content/push"))
for p in (CKPT, ART, CACHE, MAPS, PUSH):
    p.mkdir(parents=True, exist_ok=True)

def find_dataset_root(base):
    for csv_path in sorted(Path(base).rglob("train.csv")):
        d = csv_path.parent
        if (d / "train").is_dir() and (d / "test").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not locate train.csv with train/ and test/ beside it under {base}.\n"
        f"Found: {[str(p) for p in Path(base).rglob('*')][:40]}")

BASE = find_dataset_root(DATA)
TRAIN_DIR, TEST_DIR, TRAIN_CSV = BASE / "train", BASE / "test", BASE / "train.csv"
print("data root:", BASE)
print("train", len(list(TRAIN_DIR.glob('*.png'))), "| test", len(list(TEST_DIR.glob('*.png'))))

UNLABELED_DIR = next((c for c in sorted(BASE.rglob("*"))
                      if c.is_dir() and "unlabeled" in c.name.lower()), None)
print("unlabeled:", len(list(UNLABELED_DIR.glob('*.png'))) if UNLABELED_DIR else "not found",
      "(not used in Stage 8 - the SSL family already has it)")


def resolve_encoder():
    for name in [CFG.ENCODER] + list(CFG.ENCODER_FALLBACKS):
        try:
            m = timm.create_model(name, pretrained=True, features_only=True, in_chans=3)
            ch = m.feature_info.channels()
            del m; gc.collect()
            print(f"encoder ok: {name} | feature channels {ch}")
            return name
        except Exception as exc:
            print(f"  [{name}] unavailable: {type(exc).__name__}: {str(exc)[:160]}")
    raise RuntimeError("No pretrained encoder could be downloaded.")

CFG.ENCODER = resolve_encoder()
STATS["encoder"] = CFG.ENCODER

data root: /content/data/cells_segmentation
train 424 | test 182
unlabeled: 1972 (not used in Stage 8 - the SSL family already has it)


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

encoder ok: convnext_tiny.fb_in22k_ft_in1k | feature channels [96, 192, 384, 768]


In [6]:
# ── 4 · Labels and RLE codec (byte-identical to Stage 5/7) ────────────────────
df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id", "image_id", "imageid"))
ANN_COL = next(c for c in df_raw.columns
               if c.lower() in ("annotation", "encodedpixels", "rle", "segmentation"))

TRAIN_IDS = sorted(p.stem for p in TRAIN_DIR.glob("*.png"))
TEST_IDS = sorted(p.stem for p in TEST_DIR.glob("*.png"))
with Image.open(TRAIN_DIR / f"{TRAIN_IDS[0]}.png") as im:
    CFG.IMG_W, CFG.IMG_H = im.size
PAD_H = int(math.ceil(CFG.IMG_H / CFG.PAD_MULT) * CFG.PAD_MULT)
PAD_W = int(math.ceil(CFG.IMG_W / CFG.PAD_MULT) * CFG.PAD_MULT)
NPIX = CFG.IMG_H * CFG.IMG_W
print(f"image {CFG.IMG_H}x{CFG.IMG_W} -> padded {PAD_H}x{PAD_W} | "
      f"{len(TRAIN_IDS)} train / {len(TEST_IDS)} test")


def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() == "-1":
        return np.empty(0, dtype=np.int64)
    values = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = values[0::2] - 1, values[1::2]
    idx = np.concatenate([np.arange(s, s + l) for s, l in zip(starts, lengths)])
    if order == "C":
        return idx
    return (idx % h) * w + (idx // h)


def rle_encode(mask_flat_u8):
    pixels = np.concatenate([[0], mask_flat_u8, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)


def flat_to_rle(flat):
    mask = np.zeros(NPIX, dtype=np.uint8)
    mask[flat] = 1
    return rle_encode(mask)


probe = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
order_report = {}
for order in ("C", "F"):
    single = []
    for rle in probe[ANN_COL].astype(str):
        flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, order)
        if flat.size == 0 or flat.max() >= NPIX:
            continue
        m = np.zeros(NPIX, bool); m[flat] = True
        single.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    order_report[order] = float(np.mean(single) * 100) if single else 0.0
RLE_ORDER = max(order_report, key=lambda o: order_report[o])
print("single-component % by order:", {k: round(v, 1) for k, v in order_report.items()},
      "-> RLE_ORDER =", RLE_ORDER)
assert order_report[RLE_ORDER] > 90, "RLE order detection failed."

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if flat.size:
        INST[image_id].append(np.unique(flat))
counts = np.array([len(INST[i]) for i in TRAIN_IDS])
STATS["gt_mean_instances"] = float(counts.mean())
print(f"instances {int(counts.sum())} | per image mean {counts.mean():.1f}")

bad = 0
for image_id in TRAIN_IDS[:40]:
    for flat in INST[image_id]:
        back = rle_decode_flat(flat_to_rle(flat), CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
        if not np.array_equal(np.unique(back), flat):
            bad += 1
assert bad == 0, f"RLE round-trip lossy on {bad} instances."
print(">>> RLE round-trip exact on 40 images.")

image 520x704 -> padded 544x704 | 424 train / 182 test
single-component % by order: {'C': 91.5, 'F': 0.0} -> RLE_ORDER = C
instances 53065 | per image mean 125.2
>>> RLE round-trip exact on 40 images.


In [7]:
# ── 5 · Images and per-erosion target sets ────────────────────────────────────
def load_gray_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

IMAGES = {i: load_gray_u8(TRAIN_DIR / f"{i}.png") for i in TRAIN_IDS}
MEAN = float(np.mean([IMAGES[i].mean() for i in TRAIN_IDS]) / 255.0)
STD = float(np.mean([IMAGES[i].std() for i in TRAIN_IDS]) / 255.0)
print(f"loaded {len(IMAGES)} train images | mean/std {MEAN:.4f}/{STD:.4f}")


def build_targets_for(image_id, frac):
    # ch0 foreground union, ch1 eroded seed, ch2 contact zone
    h, w = CFG.IMG_H, CFG.IMG_W
    fg = np.zeros((h, w), np.uint8)
    seed = np.zeros((h, w), np.uint8)
    overlap = np.zeros((h, w), np.uint16)
    for flat in INST[image_id]:
        ys, xs = np.divmod(flat, w)
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        sub = np.zeros((y1 - y0, x1 - x0), bool)
        sub[ys - y0, xs - x0] = True
        fg[y0:y1, x0:x1] |= sub
        dist = distance_transform_edt(np.pad(sub, 1))[1:-1, 1:-1]
        dmax = dist.max()
        if dmax > 0:
            core = dist > frac * dmax
            if not core.any():
                core = dist >= dmax
            seed[y0:y1, x0:x1] |= core
        pad = CFG.BORDER_DILATION
        ys0, ys1 = max(0, y0 - pad), min(h, y1 + pad)
        xs0, xs1 = max(0, x0 - pad), min(w, x1 + pad)
        big = np.zeros((ys1 - ys0, xs1 - xs0), bool)
        big[y0 - ys0:y1 - ys0, x0 - xs0:x1 - xs0] = sub
        overlap[ys0:ys1, xs0:xs1] += binary_dilation(big, iterations=pad).astype(np.uint16)
    return np.stack([fg, seed, (overlap >= 2).astype(np.uint8)], 0)


TARGETS = None          # swapped per member; CellDataset reads this global
_TARGET_KEY = None


def load_targets(frac):
    # Each member has its own seed target, so each needs its own cache file.
    global TARGETS, _TARGET_KEY
    if _TARGET_KEY == frac:
        return
    TARGETS = None
    gc.collect()
    path = CACHE / f"targets_e{int(round(frac * 100)):03d}.npz"
    if CFG.RESUME and path.exists():
        with np.load(path) as z:
            TARGETS = {k: z[k] for k in z.files}
        print(f"targets (erosion {frac}) loaded from cache")
    else:
        t0 = time.time()
        TARGETS = {i: build_targets_for(i, frac) for i in TRAIN_IDS}
        np.savez_compressed(path, **TARGETS)
        print(f"targets (erosion {frac}) built in {time.time() - t0:.0f}s")
    _TARGET_KEY = frac
    t = TARGETS[TRAIN_IDS[0]]
    print(f"  fg {100 * t[0].mean():.1f}% | seed {100 * t[1].mean():.1f}% | "
          f"border {100 * t[2].mean():.1f}%")


load_targets(CFG.MEMBERS[0]["erosion"])

loaded 424 train images | mean/std 0.5018/0.0522
targets (erosion 0.45) built in 24s
  fg 23.0% | seed 7.7% | border 3.6%


In [8]:
# ── 6 · Grouped split + THE SPLIT ASSERT ──────────────────────────────────────
# Stage 7 was configured to reproduce Stage 5's split and did not. If this cell
# does not reproduce Stage 7's split exactly, the Stage 9 blend weight cannot be
# tuned honestly - so it stops here, three minutes in, instead of two hours in.
TH_H, TH_W = 65, 88


def thumbnail_vec(arr):
    a = np.asarray(Image.fromarray(arr).resize((TH_W, TH_H), Image.BILINEAR), np.float32).ravel()
    a -= a.mean()
    n = np.linalg.norm(a)
    return a / n if n > 0 else a

THUMBS = np.stack([thumbnail_vec(IMAGES[i]) for i in TRAIN_IDS])
CORR = THUMBS @ THUMBS.T
np.fill_diagonal(CORR, 0.0)

if CFG.GROUP_SPLIT:
    adj = CORR >= CFG.DUP_CORR_THR
    group_of = -np.ones(len(TRAIN_IDS), int)
    g = 0
    for i in range(len(TRAIN_IDS)):
        if group_of[i] >= 0:
            continue
        stack, group_of[i] = [i], g
        while stack:
            u = stack.pop()
            for v in np.nonzero(adj[u])[0]:
                if group_of[v] < 0:
                    group_of[v] = g
                    stack.append(v)
        g += 1
    n_groups = g
else:
    group_of = np.arange(len(TRAIN_IDS))
    n_groups = len(TRAIN_IDS)

rng = np.random.RandomState(CFG.SPLIT_SEED)
order = rng.permutation(n_groups)
fold_of_group = {int(gid): int(k % CFG.N_FOLDS) for k, gid in enumerate(order)}
val_mask = np.array([fold_of_group[int(x)] == CFG.FOLD for x in group_of])
train_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if not m]
val_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if m]

vi, ti = np.nonzero(val_mask)[0], np.nonzero(~val_mask)[0]
max_corr = float(CORR[np.ix_(vi, ti)].max())
print(f"groups {n_groups} | train/val {len(train_ids)}/{len(val_ids)} | "
      f"max val-to-train corr {max_corr:.3f}")
assert max_corr < CFG.DUP_CORR_THR, "Near-duplicate leaked across the split."

got = (len(train_ids), len(val_ids), n_groups)
want = (CFG.EXPECT_TRAIN, CFG.EXPECT_VAL, CFG.EXPECT_GROUPS)
if got != want:
    msg = (f"SPLIT MISMATCH: got train/val/groups {got}, Stage 7 had {want}.\n"
           "The Stage 9 blend needs both families scored on the SAME validation images.\n"
           "Set CFG.STRICT_SPLIT = False to train anyway (the models stay useful as "
           "standalone submissions), but do NOT blend the maps against Stage 7's.")
    if CFG.STRICT_SPLIT:
        raise RuntimeError(msg)
    print("!! " + msg)
else:
    print(">>> split reproduces Stage 7 exactly - the Stage 9 blend is tunable.")

# The identity artifact. Stage 9 verifies against this rather than trusting a count.
(ART / "val_ids.json").write_text(json.dumps(
    {"val_ids": val_ids, "n_train": len(train_ids), "n_groups": int(n_groups),
     "split_seed": CFG.SPLIT_SEED, "fold": CFG.FOLD, "max_val_train_corr": max_corr}, indent=1))

ALL_IDS = TRAIN_IDS + TEST_IDS
ALL_THUMBS = np.concatenate([THUMBS, np.stack([
    thumbnail_vec(load_gray_u8(TEST_DIR / f"{i}.png")) for i in TEST_IDS])])
km = KMeans(n_clusters=CFG.CLUSTER_K, n_init=10, random_state=0).fit(ALL_THUMBS)
CLUSTER = {i: int(c) for i, c in zip(ALL_IDS, km.labels_)}
STATS.update(n_groups=int(n_groups), max_val_train_corr=max_corr,
             n_train=len(train_ids), n_val=len(val_ids))

groups 421 | train/val 338/86 | max val-to-train corr 0.665
>>> split reproduces Stage 7 exactly - the Stage 9 blend is tunable.


In [9]:
# ── 7 · Dataset and augmentation (Stage 5/7 verbatim) ─────────────────────────
def resize_pair(img, tgt, scale):
    h, w = img.shape
    nh, nw = max(64, int(round(h * scale))), max(64, int(round(w * scale)))
    if HAVE_CV2:
        img_r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        tgt_r = np.stack([cv2.resize(tgt[c], (nw, nh), interpolation=cv2.INTER_NEAREST)
                          for c in range(tgt.shape[0])])
    else:
        img_r = np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize(
            (nw, nh), Image.BILINEAR), np.float32) / 255.0
        tgt_r = np.stack([np.asarray(Image.fromarray(tgt[c]).resize((nw, nh), Image.NEAREST))
                          for c in range(tgt.shape[0])])
    return img_r.astype(np.float32), tgt_r.astype(np.uint8)


def pad_to(arr, height, width, mode="reflect"):
    ph, pw = max(0, height - arr.shape[-2]), max(0, width - arr.shape[-1])
    if ph == 0 and pw == 0:
        return arr
    pad_width = [(0, 0)] * (arr.ndim - 2) + [(0, ph), (0, pw)]
    kw = {} if mode == "reflect" else {"constant_values": 0}
    return np.pad(arr, pad_width, mode=mode, **kw)


class CellDataset(Dataset):
    def __init__(self, ids, crop, training=True):
        self.ids = list(ids)
        self.crop_h, self.crop_w = crop
        self.training = bool(training)

    def __len__(self):
        return len(self.ids)

    def augment(self, img, tgt):
        if random.random() < CFG.AUG_HFLIP:
            img, tgt = img[:, ::-1], tgt[:, :, ::-1]
        if random.random() < CFG.AUG_VFLIP:
            img, tgt = img[::-1], tgt[:, ::-1]
        img = np.ascontiguousarray(img)
        tgt = np.ascontiguousarray(tgt)
        s = random.uniform(*CFG.AUG_SCALE)
        if abs(s - 1.0) > 1e-3:
            img, tgt = resize_pair(img, tgt, s)
        img = img + random.uniform(-CFG.AUG_BRIGHT, CFG.AUG_BRIGHT) * STD
        f = 1.0 + random.uniform(-CFG.AUG_CONTRAST, CFG.AUG_CONTRAST)
        img = (img - MEAN) * f + MEAN
        img = img + np.random.randn(*img.shape).astype(np.float32) * CFG.AUG_NOISE * STD
        return np.clip(img, 0.0, 1.0).astype(np.float32), tgt

    def random_crop(self, img, tgt):
        img = pad_to(img, self.crop_h, self.crop_w)
        tgt = pad_to(tgt, self.crop_h, self.crop_w, mode="constant")
        h, w = img.shape
        y = random.randint(0, h - self.crop_h)
        x = random.randint(0, w - self.crop_w)
        return img[y:y + self.crop_h, x:x + self.crop_w], tgt[:, y:y + self.crop_h, x:x + self.crop_w]

    def __getitem__(self, index):
        image_id = self.ids[index]
        img = IMAGES[image_id].astype(np.float32) / 255.0
        tgt = TARGETS[image_id].copy()
        if self.training:
            img, tgt = self.augment(img, tgt)
            img, tgt = self.random_crop(img, tgt)
        else:
            img = pad_to(img, PAD_H, PAD_W)
            tgt = pad_to(tgt, PAD_H, PAD_W, mode="constant")
        weight = 1.0 + CFG.BORDER_WEIGHT * tgt[2].astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).repeat(3, 1, 1)
        y = torch.from_numpy(np.ascontiguousarray(tgt[:2]).astype(np.float32))
        w = torch.from_numpy(np.ascontiguousarray(weight)).unsqueeze(0)
        return x, y, w, image_id


def normalize_batch(x):
    return (x - MEAN) / max(STD, 1e-6)


_ds = CellDataset(train_ids, CFG.MEMBERS[0]["crop"], True)
_x, _y, _w, _ = _ds[0]
print("sample shapes:", tuple(_x.shape), tuple(_y.shape), "| weight range",
      float(_w.min()), float(_w.max()))
del _ds, _x, _y, _w

sample shapes: (3, 448, 576) (2, 448, 576) | weight range 1.0 5.0


In [10]:
# ── 8 · ConvNeXt-UNet ─────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


def up_to(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self, encoder_name, out_ch=2, dec=(256, 128, 64, 32), pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True, in_chans=3)
        chs = self.encoder.feature_info.channels()[-4:]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dec[3], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec[3]), nn.ReLU(inplace=True))
        self.d3 = ConvBlock(chs[3] + chs[2], dec[0])
        self.d2 = ConvBlock(dec[0] + chs[1], dec[1])
        self.d1 = ConvBlock(dec[1] + chs[0], dec[2])
        self.d0 = ConvBlock(dec[2], dec[3])
        self.dfull = ConvBlock(dec[3] * 2, dec[3])
        self.head = nn.Conv2d(dec[3], out_ch, 1)

    def forward(self, x):
        stem = self.stem(x)
        f1, f2, f3, f4 = self.encoder(x)[-4:]
        d = self.d3(torch.cat([up_to(f4, f3), f3], 1))
        d = self.d2(torch.cat([up_to(d, f2), f2], 1))
        d = self.d1(torch.cat([up_to(d, f1), f1], 1))
        d = self.d0(F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False))
        d = self.dfull(torch.cat([up_to(d, stem), stem], 1))
        return self.head(d)


def loss_fn(logits, target, weight):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (bce * weight).mean()
    prob = torch.sigmoid(logits)
    num = 2.0 * (prob * target).sum(dim=(2, 3)) + 1.0
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1.0
    return bce + (1.0 - (num / den).mean())


_m = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE).eval()
with torch.no_grad(), amp_autocast():
    _o = _m(torch.zeros(1, 3, PAD_H, PAD_W, device=DEVICE))
print("model forward ok:", tuple(_o.shape), "| params",
      f"{sum(p.numel() for p in _m.parameters()) / 1e6:.1f}M")
assert _o.shape[-2:] == (PAD_H, PAD_W)
del _m, _o; gc.collect(); torch.cuda.empty_cache()

model forward ok: (1, 2, 544, 704) | params 32.0M


In [11]:
# ── 9 · Decode, official metric, fast threshold evaluation ────────────────────
def watershed_labels(fg, seed, t_cell, t_seed):
    mask = fg >= t_cell
    if not mask.any():
        return np.zeros(fg.shape, np.int32), 0
    markers, n = ndi.label((seed >= t_seed) & mask)
    if n == 0:
        markers, n = ndi.label(mask)
        if n == 0:
            return np.zeros(fg.shape, np.int32), 0
    lab = watershed(-fg, markers, mask=mask).astype(np.int32)
    return lab, int(lab.max())


def label_stats(lab, fg, seed, nlab):
    flat = lab.ravel()
    area = np.bincount(flat, minlength=nlab + 1).astype(np.float64)
    denom = np.maximum(area, 1.0)
    mean_fg = np.bincount(flat, weights=fg.ravel(), minlength=nlab + 1) / denom
    mean_seed = np.bincount(flat, weights=seed.ravel(), minlength=nlab + 1) / denom
    return area, mean_fg, mean_seed


def prepare_eval(fg, seed, gt_flats, t_cell, t_seed):
    lab, nlab = watershed_labels(fg, seed, t_cell, t_seed)
    area, mean_fg, mean_seed = label_stats(lab, fg, seed, nlab)
    n_gt = len(gt_flats)
    if nlab == 0 or n_gt == 0:
        return dict(n_gt=n_gt, area=area, score=mean_seed, gi=np.empty(0, int),
                    pi=np.empty(0, int), lab=lab, nlab=nlab)
    flatlab = lab.ravel()
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    C = np.zeros((n_gt, nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(flatlab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    return dict(n_gt=n_gt, area=area, score=mean_seed, gi=gi[o], pi=pi[o], lab=lab, nlab=nlab)


def f1_filtered(prep, min_area, score_thr):
    # 2TP / (2TP + FP + FN), greedy one-to-one at Dice >= 0.50, mean over images.
    keep = (prep["area"] >= min_area) & (prep["score"] >= score_thr)
    if keep.size:
        keep[0] = False
    n_pred = int(keep.sum())
    n_gt = prep["n_gt"]
    if n_gt == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0:
        return 0.0
    used_g = np.zeros(n_gt, bool)
    used_p = np.zeros(keep.size, bool)
    tp = 0
    for g, p in zip(prep["gi"], prep["pi"]):
        if not keep[p] or used_g[g] or used_p[p]:
            continue
        used_g[g] = True; used_p[p] = True; tp += 1
    return 2.0 * tp / (2.0 * tp + (n_pred - tp) + (n_gt - tp))


def instances_from_maps(fg, seed, post):
    lab, nlab = watershed_labels(fg, seed, post["t_cell"], post["t_seed"])
    if nlab == 0:
        return []
    area, _, mean_seed = label_stats(lab, fg, seed, nlab)
    keep = (area >= post["min_area"]) & (mean_seed >= post.get("score_thr", 0.0))
    keep[0] = False
    flat = lab.ravel()
    order_ = np.argsort(flat, kind="stable")
    sorted_lab = flat[order_]
    bounds = np.searchsorted(sorted_lab, np.arange(nlab + 2))
    out = []
    for j in np.nonzero(keep)[0]:
        idx = order_[bounds[j]:bounds[j + 1]]
        if idx.size:
            out.append(np.sort(idx))
    return out


print("decode + metric ready")

decode + metric ready


In [12]:
# ── 10 · Inference with flip + scale TTA (identical to Stage 7) ───────────────
@torch.no_grad()
def predict_maps(model, ids, image_dir, scales=None, flips=None):
    model.eval()
    scales = CFG.TTA_SCALES if scales is None else scales
    flips = CFG.TTA_FLIPS if flips is None else flips
    out = {}
    for image_id in ids:
        arr = IMAGES[image_id] if image_dir is None else load_gray_u8(image_dir / f"{image_id}.png")
        base = arr.astype(np.float32) / 255.0
        acc = np.zeros((2, CFG.IMG_H, CFG.IMG_W), np.float32)
        n = 0
        for s in scales:
            if abs(s - 1.0) < 1e-3:
                img = base
            elif HAVE_CV2:
                img = cv2.resize(base, (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))),
                                 interpolation=cv2.INTER_LINEAR)
            else:
                img = np.asarray(Image.fromarray((base * 255).astype(np.uint8)).resize(
                    (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))), Image.BILINEAR),
                    np.float32) / 255.0
            h, w = img.shape
            ph = int(math.ceil(h / CFG.PAD_MULT) * CFG.PAD_MULT)
            pw = int(math.ceil(w / CFG.PAD_MULT) * CFG.PAD_MULT)
            x = torch.from_numpy(np.ascontiguousarray(pad_to(img, ph, pw)))
            x = x.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            variants = [(False, False)] if not flips else [(False, False), (True, False),
                                                           (False, True), (True, True)]
            for hf, vf in variants:
                xin = x
                if hf:
                    xin = torch.flip(xin, dims=[3])
                if vf:
                    xin = torch.flip(xin, dims=[2])
                with amp_autocast():
                    logits = model(xin)
                prob = torch.sigmoid(logits.float())
                if vf:
                    prob = torch.flip(prob, dims=[2])
                if hf:
                    prob = torch.flip(prob, dims=[3])
                p = prob[0, :, :h, :w].cpu().numpy()
                if p.shape[1:] != (CFG.IMG_H, CFG.IMG_W):
                    if HAVE_CV2:
                        p = np.stack([cv2.resize(p[c], (CFG.IMG_W, CFG.IMG_H),
                                                 interpolation=cv2.INTER_LINEAR) for c in range(2)])
                    else:
                        p = np.stack([np.asarray(Image.fromarray((p[c] * 255).astype(np.uint8)).resize(
                            (CFG.IMG_W, CFG.IMG_H), Image.BILINEAR), np.float32) / 255.0
                            for c in range(2)])
                acc += p
                n += 1
        out[image_id] = np.clip(acc / max(n, 1) * 255.0, 0, 255).astype(np.uint8)
    return out


def save_maps(tag, split, maps):
    path = MAPS / f"{tag}_{split}.npz"
    np.savez_compressed(path, **maps)
    return path


def load_maps(tag, split):
    with np.load(MAPS / f"{tag}_{split}.npz") as z:
        return {k: z[k] for k in z.files}


def maps_to_float(u8):
    return u8[0].astype(np.float32) / 255.0, u8[1].astype(np.float32) / 255.0


print("inference helpers ready")

inference helpers ready


In [13]:
# ── 11 · Push artifacts to a Kaggle dataset ───────────────────────────────────
# This is what makes Colab non-lossy. Called after EVERY member, so a disconnect
# at minute 95 still leaves n1's checkpoint and maps safely on Kaggle.
DATASET_ID = f"{KAGGLE_USER}/{CFG.DATASET_SLUG}"
_DATASET_EXISTS = [False]


def stage_for_push(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            shutil.copy2(p, PUSH / p.name)


def push_dataset(message):
    if not CFG.PUSH_DATASET:
        return
    files = sorted(p for p in PUSH.iterdir() if p.name != "dataset-metadata.json")
    if not files:
        print("push: nothing staged")
        return
    (PUSH / "dataset-metadata.json").write_text(json.dumps(
        {"title": CFG.DATASET_TITLE, "id": DATASET_ID,
         "licenses": [{"name": "CC0-1.0"}]}, indent=1))
    total = sum(p.stat().st_size for p in files) / 1e6
    print(f"push -> {DATASET_ID} | {len(files)} files, {total:.0f} MB | {message}")
    cmd = (["kaggle", "datasets", "version", "-p", str(PUSH), "-m", message, "-r", "skip"]
           if _DATASET_EXISTS[0] else
           ["kaggle", "datasets", "create", "-p", str(PUSH), "-r", "skip"])
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print("  " + out[-500:].replace("\n", "\n  "))
    if r.returncode != 0 and not _DATASET_EXISTS[0]:
        # Already exists from an earlier session - switch to versioning and retry once.
        _DATASET_EXISTS[0] = True
        r = subprocess.run(["kaggle", "datasets", "version", "-p", str(PUSH),
                            "-m", message, "-r", "skip"], capture_output=True, text=True)
        print("  retry:", (r.stdout + r.stderr).strip()[-500:])
    if r.returncode == 0:
        _DATASET_EXISTS[0] = True


stage_for_push([ART / "val_ids.json"])
push_dataset("stage 8 - split identity")

push -> ghostylicious/stage-8-nonssl | 1 files, 0 MB | stage 8 - split identity
  Starting upload for file val_ids.json
  Upload successful: val_ids.json (2KB)
  Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ghostylicious/stage-8-nonssl
  
    0%|          | 0.00/1.64k [00:00<?, ?B/s]
  100%|██████████| 1.64k/1.64k [00:00<00:00, 2.56kB/s]


In [14]:
# ── 12 · Train one member (plain ImageNet init - no SSL encoder) ──────────────
def quick_val_f1(model, ids, post):
    maps = predict_maps(model, ids, None, scales=(1.0,), flips=False)
    scores = []
    for image_id in ids:
        fg, seed = maps_to_float(maps[image_id])
        prep = prepare_eval(fg, seed, INST[image_id], post["t_cell"], post["t_seed"])
        scores.append(f1_filtered(prep, post["min_area"], post["score_thr"]))
    return float(np.mean(scores))


def train_member(spec):
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    best_path = CKPT / f"{tag}_best.pt"
    if CFG.RESUME and best_path.exists():
        blob = torch.load(best_path, map_location="cpu")
        if blob.get("run_tag") == CFG.RUN_TAG and blob.get("member") == spec["name"]:
            print(f"[{spec['name']}] finished checkpoint found (epoch {blob['epoch']}, "
                  f"F1 {blob['f1']:.4f}) - skipping training")
            return blob["f1"]

    load_targets(spec["erosion"])
    seed_everything(spec["seed"])
    # pretrained=True gives plain ImageNet-21k->1k weights. Nothing else is loaded:
    # this member's whole purpose is to NOT share the SSL encoder with m1/m2/m3.
    model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)

    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW(
        [{"params": dec_params, "lr": CFG.SUP_LR},
         {"params": enc_params, "lr": CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT}],
        weight_decay=CFG.SUP_WD)

    loader = DataLoader(CellDataset(train_ids, spec["crop"], True), batch_size=CFG.SUP_BATCH,
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
                        persistent_workers=True)
    total_steps = CFG.SUP_EPOCHS * CFG.STEPS_PER_EPOCH
    scaler = make_scaler()
    best_f1, best_epoch, log = -1.0, -1, []
    t0 = time.time()

    def lr_at(step):
        if step < CFG.WARMUP_STEPS:
            return (step + 1) / max(1, CFG.WARMUP_STEPS)
        p = (step - CFG.WARMUP_STEPS) / max(1, total_steps - CFG.WARMUP_STEPS)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

    step = 0
    it = iter(loader)
    for ep in range(CFG.SUP_EPOCHS):
        if time.time() - t0 > CFG.MEMBER_BUDGET_SEC:
            print(f"[{spec['name']}] member budget reached at epoch {ep} - stopping")
            break
        freeze = ep < CFG.FREEZE_ENCODER_EPOCHS
        for p in enc_params:
            p.requires_grad_(not freeze)
        model.train()
        run = 0.0
        for _ in range(CFG.STEPS_PER_EPOCH):
            try:
                x, y, w, _ = next(it)
            except StopIteration:
                it = iter(loader)
                x, y, w, _ = next(it)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            w = w.to(DEVICE, non_blocking=True)
            mult = lr_at(step)
            opt.param_groups[0]["lr"] = CFG.SUP_LR * mult
            opt.param_groups[1]["lr"] = (CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT * mult
                                         * (0.0 if freeze else 1.0))
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = loss_fn(model(x), y, w)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
        msg = f"[{spec['name']}] ep {ep + 1:02d}/{CFG.SUP_EPOCHS} loss {run / CFG.STEPS_PER_EPOCH:.4f}"
        if ep + 1 >= CFG.EVAL_START and (ep + 1 - CFG.EVAL_START) % CFG.EVAL_EVERY == 0:
            f1 = quick_val_f1(model, val_ids, CFG.DEFAULT_POST)
            msg += f" | val F1 {f1:.4f}"
            if f1 > best_f1:
                best_f1, best_epoch = f1, ep + 1
                torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": ep + 1,
                            "f1": f1, "state_dict": model.state_dict(), "spec": spec,
                            "encoder": CFG.ENCODER}, best_path)
                msg += "  *"
            log.append(dict(member=spec["name"], epoch=ep + 1,
                            loss=run / CFG.STEPS_PER_EPOCH, f1=f1))
        print(msg + f" | {fmt(time.time() - t0)}", flush=True)

    if best_f1 < 0:
        best_f1, best_epoch = quick_val_f1(model, val_ids, CFG.DEFAULT_POST), CFG.SUP_EPOCHS
        torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": best_epoch,
                    "f1": best_f1, "state_dict": model.state_dict(), "spec": spec,
                    "encoder": CFG.ENCODER}, best_path)
    pd.DataFrame(log).to_csv(ART / f"train_log_{spec['name']}.csv", index=False)
    print(f"[{spec['name']}] best epoch {best_epoch} | quick val F1 {best_f1:.4f} | "
          f"{fmt(time.time() - t0)}")
    del model, opt, loader, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f1

In [15]:
# ── 13 · Run members, cache maps, write a per-member submission, push ─────────
def write_submission(path, maps, ids, post, per_image_post=None):
    rows, counts, fallbacks = [], [], 0
    for image_id in ids:
        fg, seed = maps_to_float(maps[image_id]) if maps[image_id].dtype == np.uint8 \
            else (maps[image_id][0], maps[image_id][1])
        use = post if per_image_post is None else per_image_post.get(image_id, post)
        inst = instances_from_maps(fg, seed, use)
        if not inst:  # an image with GT and zero masks scores exactly 0.0
            fallbacks += 1
            inst = instances_from_maps(fg, seed, dict(t_cell=0.30, t_seed=0.30,
                                                      min_area=0, score_thr=0.0))[:1]
            if not inst:
                h = CFG.IMG_H // 2
                inst = [np.arange(h * CFG.IMG_W, h * CFG.IMG_W + 25)]
        counts.append(len(inst))
        rows.extend((image_id, flat_to_rle(f)) for f in inst)
    frame = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
    frame.insert(0, CFG.SUB_INDEX_COL, np.arange(len(frame), dtype=np.int64))
    frame.to_csv(path, index=False)
    assert set(frame[ID_COL].astype(str)) == set(ids), "Submission is missing test images."
    assert frame[ANN_COL].notna().all() and frame[CFG.SUB_INDEX_COL].is_unique
    return dict(path=str(path), rows=len(frame), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=fallbacks)


MEMBER_RESULTS = []
for spec in CFG.MEMBERS:
    if elapsed() > CFG.TOTAL_BUDGET_SEC - CFG.FINAL_RESERVE_SEC:
        print(f"!! stopping after {len(MEMBER_RESULTS)} member(s) - reserving time for the sweep")
        break
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    quick = train_member(spec)

    if CFG.RESUME and (MAPS / f"{tag}_val.npz").exists() and (MAPS / f"{tag}_test.npz").exists():
        print(f"[{spec['name']}] cached maps found - skipping inference")
    else:
        blob = torch.load(CKPT / f"{tag}_best.pt", map_location="cpu")
        assert blob["run_tag"] == CFG.RUN_TAG and blob["member"] == spec["name"], \
            "Checkpoint identity mismatch."
        model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS,
                     pretrained=False).to(DEVICE)
        model.load_state_dict(blob["state_dict"])
        t0 = time.time()
        save_maps(tag, "val", predict_maps(model, val_ids, None))
        save_maps(tag, "test", predict_maps(model, TEST_IDS, TEST_DIR))
        print(f"[{spec['name']}] TTA maps cached in {fmt(time.time() - t0)}")
        del model, blob
        gc.collect(); torch.cuda.empty_cache()

    vmaps = load_maps(tag, "val")
    tta_f1 = float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(vmaps[i]), INST[i],
                                 CFG.DEFAULT_POST["t_cell"], CFG.DEFAULT_POST["t_seed"]),
                    CFG.DEFAULT_POST["min_area"], CFG.DEFAULT_POST["score_thr"])
        for i in val_ids]))
    info = write_submission(ART / f"submission_{spec['name']}.csv",
                            load_maps(tag, "test"), TEST_IDS, CFG.DEFAULT_POST)
    print(f"[{spec['name']}] TTA val F1 (default post) {tta_f1:.4f} | "
          f"test preds/image {info['mean']:.1f}")
    MEMBER_RESULTS.append(dict(member=spec["name"], erosion=spec["erosion"],
                               quick_f1=quick, tta_f1=tta_f1, **info))
    del vmaps
    gc.collect()

    # Safety: everything this member produced goes to Kaggle before the next starts.
    stage_for_push([CKPT / f"{tag}_best.pt", MAPS / f"{tag}_val.npz", MAPS / f"{tag}_test.npz",
                    ART / f"submission_{spec['name']}.csv", ART / f"train_log_{spec['name']}.csv"])
    push_dataset(f"stage 8 - member {spec['name']} complete")

STATS["members"] = MEMBER_RESULTS
assert MEMBER_RESULTS, "No member finished."
print("\nmembers complete:", [m["member"] for m in MEMBER_RESULTS], "| elapsed", fmt(elapsed()))

[n1] ep 01/70 loss 1.6338 | 0h 00m 38s
[n1] ep 02/70 loss 1.4678 | 0h 00m 50s
[n1] ep 03/70 loss 1.3828 | 0h 01m 24s
[n1] ep 04/70 loss 1.3082 | 0h 01m 40s
[n1] ep 05/70 loss 1.2410 | 0h 01m 57s
[n1] ep 06/70 loss 1.1756 | 0h 02m 14s
[n1] ep 07/70 loss 1.1204 | 0h 02m 31s
[n1] ep 08/70 loss 1.0625 | 0h 02m 48s
[n1] ep 09/70 loss 1.0127 | 0h 03m 06s
[n1] ep 10/70 loss 0.9581 | 0h 03m 23s
[n1] ep 11/70 loss 0.9165 | 0h 03m 41s
[n1] ep 12/70 loss 0.8726 | 0h 03m 58s
[n1] ep 13/70 loss 0.8414 | 0h 04m 16s
[n1] ep 14/70 loss 0.8050 | 0h 04m 33s
[n1] ep 15/70 loss 0.7590 | 0h 04m 50s
[n1] ep 16/70 loss 0.7242 | 0h 05m 08s
[n1] ep 17/70 loss 0.6984 | 0h 05m 25s
[n1] ep 18/70 loss 0.6711 | 0h 05m 43s
[n1] ep 19/70 loss 0.6446 | 0h 06m 00s
[n1] ep 20/70 loss 0.6158 | 0h 06m 18s
[n1] ep 21/70 loss 0.5920 | 0h 06m 35s
[n1] ep 22/70 loss 0.5665 | 0h 06m 53s
[n1] ep 23/70 loss 0.5462 | 0h 07m 10s
[n1] ep 24/70 loss 0.5255 | val F1 0.7313  * | 0h 07m 34s
[n1] ep 25/70 loss 0.5090 | 0h 07m 51s
[n1] e

[n2] ep 01/70 loss 1.5325 | 0h 00m 35s
[n2] ep 02/70 loss 1.3880 | 0h 00m 44s
[n2] ep 03/70 loss 1.2884 | 0h 01m 15s
[n2] ep 04/70 loss 1.2235 | 0h 01m 29s
[n2] ep 05/70 loss 1.1593 | 0h 01m 43s
[n2] ep 06/70 loss 1.1053 | 0h 01m 56s
[n2] ep 07/70 loss 1.0505 | 0h 02m 10s
[n2] ep 08/70 loss 1.0022 | 0h 02m 24s
[n2] ep 09/70 loss 0.9611 | 0h 02m 37s
[n2] ep 10/70 loss 0.9287 | 0h 02m 51s
[n2] ep 11/70 loss 0.8818 | 0h 03m 05s
[n2] ep 12/70 loss 0.8429 | 0h 03m 19s
[n2] ep 13/70 loss 0.8088 | 0h 03m 32s
[n2] ep 14/70 loss 0.7674 | 0h 03m 46s
[n2] ep 15/70 loss 0.7285 | 0h 04m 00s
[n2] ep 16/70 loss 0.6954 | 0h 04m 14s
[n2] ep 17/70 loss 0.6597 | 0h 04m 27s
[n2] ep 18/70 loss 0.6257 | 0h 04m 41s
[n2] ep 19/70 loss 0.5891 | 0h 04m 55s
[n2] ep 20/70 loss 0.5685 | 0h 05m 09s
[n2] ep 21/70 loss 0.5435 | 0h 05m 23s
[n2] ep 22/70 loss 0.5291 | 0h 05m 37s
[n2] ep 23/70 loss 0.5102 | 0h 05m 50s
[n2] ep 24/70 loss 0.4943 | val F1 0.7663  * | 0h 06m 09s
[n2] ep 25/70 loss 0.4796 | 0h 06m 23s
[n2] e

In [16]:
# ── 14 · Sweep each source and their average, pick the best ───────────────────
# n1 and n2 were trained against DIFFERENT seed targets, so their seed channels are
# not the same quantity. Averaging them may or may not beat either alone - that is
# measured here rather than assumed.
ACTIVE = [m["member"] for m in MEMBER_RESULTS]

def build_val(names):
    acc = None
    for n in names:
        mp = load_maps(f"{CFG.RUN_TAG}_{n}", "val")
        if acc is None:
            acc = {k: mp[k].astype(np.float32) for k in mp}
        else:
            for k in acc:
                acc[k] += mp[k].astype(np.float32)
        del mp; gc.collect()
    for k in acc:
        acc[k] = acc[k] / len(names) / 255.0
    return acc


def sweep(maps, image_ids, t_cell_grid, t_seed_grid, verbose=False):
    rows, per_image = [], {}
    t0 = time.time()
    for tc in t_cell_grid:
        for ts in t_seed_grid:
            preps = [prepare_eval(maps[i][0], maps[i][1], INST[i], tc, ts) for i in image_ids]
            for ma in CFG.MIN_AREA_GRID:
                for st in CFG.SCORE_THR_GRID:
                    vals = np.array([f1_filtered(p, ma, st) for p in preps])
                    rows.append(dict(t_cell=tc, t_seed=ts, min_area=ma, score_thr=st,
                                     f1=float(vals.mean()),
                                     mean_pred=float(np.mean([
                                         int(((p["area"][1:] >= ma) & (p["score"][1:] >= st)).sum())
                                         for p in preps]))))
                    per_image[(tc, ts, ma, st)] = vals
            del preps
        if verbose:
            print(f"    t_cell {tc:.2f} ({time.time() - t0:.0f}s)", flush=True)
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True), per_image


def plateau_post(fine):
    peak = float(fine.f1.max())
    plateau = fine[fine.f1 >= peak - CFG.ROBUST_TOL]
    snap = lambda v, grid: min(grid, key=lambda g: abs(g - v))
    post = dict(t_cell=snap(float(np.median(plateau.t_cell)), sorted(set(fine.t_cell))),
                t_seed=snap(float(np.median(plateau.t_seed)), sorted(set(fine.t_seed))),
                min_area=int(snap(int(np.median(plateau.min_area)), CFG.MIN_AREA_GRID)),
                score_thr=snap(float(np.median(plateau.score_thr)), CFG.SCORE_THR_GRID))
    return post


SOURCES = [(n, [n]) for n in ACTIVE]
if len(ACTIVE) > 1:
    SOURCES.append(("ens_" + "_".join(ACTIVE), ACTIVE))

sub_ids = val_ids[:CFG.SWEEP_SUBSET] if len(val_ids) > CFG.SWEEP_SUBSET else val_ids
RESULTS = {}
for label, names in SOURCES:
    print(f"\n=== sweeping {label} ===", flush=True)
    vmaps = build_val(names)
    coarse, _ = sweep(vmaps, sub_ids, CFG.T_CELL_GRID, CFG.T_SEED_GRID, verbose=True)
    top = coarse.head(CFG.SWEEP_REFINE_TOP)
    fine, per_image = sweep(vmaps, val_ids,
                            sorted(set(top.t_cell)), sorted(set(top.t_seed)), verbose=False)
    post = plateau_post(fine)
    key = (post["t_cell"], post["t_seed"], post["min_area"], post["score_thr"])
    if key not in per_image:
        b = fine.iloc[0]
        post = dict(t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                    min_area=int(b.min_area), score_thr=float(b.score_thr))
        key = (post["t_cell"], post["t_seed"], post["min_area"], post["score_thr"])
    f1 = float(per_image[key].mean())
    RESULTS[label] = dict(names=names, post=post, f1=f1, fine=fine,
                          per_image=per_image, key=key, vmaps=vmaps)
    print(f"{label}: local F1 {f1:.4f} | {post} | peak {fine.f1.max():.4f}")
    fine.to_csv(ART / f"sweep_{label}.csv", index=False)
    if label != SOURCES[-1][0]:
        del vmaps, RESULTS[label]["vmaps"]
        RESULTS[label]["vmaps"] = None
        gc.collect()

BEST_LABEL = max(RESULTS, key=lambda k: RESULTS[k]["f1"])
BEST = RESULTS[BEST_LABEL]
print(f"\n>>> best source: {BEST_LABEL} (local F1 {BEST['f1']:.4f})")
print("    n1 0.45-erosion control vs n2 0.58-erosion probe vs their average - "
      "whichever wins here is the one that goes to Stage 9.")
STATS["sources"] = {k: dict(f1=v["f1"], post=v["post"]) for k, v in RESULTS.items()}
STATS["best_source"] = BEST_LABEL


=== sweeping n1 ===
    t_cell 0.40 (8s)
    t_cell 0.45 (16s)
    t_cell 0.50 (23s)
    t_cell 0.55 (31s)
    t_cell 0.60 (38s)
n1: local F1 0.7962 | {'t_cell': 0.4, 't_seed': 0.7, 'min_area': 90, 'score_thr': 0.0} | peak 0.7962

=== sweeping n2 ===
    t_cell 0.40 (8s)
    t_cell 0.45 (16s)
    t_cell 0.50 (24s)
    t_cell 0.55 (31s)
    t_cell 0.60 (39s)
n2: local F1 0.8055 | {'t_cell': 0.6, 't_seed': 0.7, 'min_area': 70, 'score_thr': 0.0} | peak 0.8075

=== sweeping ens_n1_n2 ===
    t_cell 0.40 (8s)
    t_cell 0.45 (16s)
    t_cell 0.50 (24s)
    t_cell 0.55 (32s)
    t_cell 0.60 (39s)
ens_n1_n2: local F1 0.8073 | {'t_cell': 0.5, 't_seed': 0.75, 'min_area': 70, 'score_thr': 0.0} | peak 0.8090

>>> best source: ens_n1_n2 (local F1 0.8073)
    n1 0.45-erosion control vs n2 0.58-erosion probe vs their average - whichever wins here is the one that goes to Stage 9.


In [17]:
# ── 15 · RLE parity on the actual predictions ─────────────────────────────────
vmaps = BEST["vmaps"] if BEST["vmaps"] is not None else build_val(BEST["names"])
POST = BEST["post"]
parity = []
for image_id in val_ids[:25]:
    inst = instances_from_maps(vmaps[image_id][0], vmaps[image_id][1], POST)
    if not inst:
        parity.append(0.0)
        continue
    round_trip = [rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER) for f in inst]
    lab = np.zeros(NPIX, np.int32)
    for j, f in enumerate(round_trip, 1):
        lab[f] = j
    nlab = len(round_trip)
    gt_flats = INST[image_id]
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    area = np.bincount(lab, minlength=nlab + 1).astype(np.float64)
    C = np.zeros((len(gt_flats), nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(lab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    used_g, used_p, tp = set(), set(), 0
    for g, p in zip(gi[o], pi[o]):
        if g in used_g or p in used_p:
            continue
        used_g.add(g); used_p.add(p); tp += 1
    parity.append(2.0 * tp / (2.0 * tp + (nlab - tp) + (len(gt_flats) - tp)))

direct = float(BEST["per_image"][BEST["key"]][:25].mean())
delta = abs(direct - float(np.mean(parity)))
print(f"parity on 25 images | in-memory {direct:.4f} | after RLE {np.mean(parity):.4f} | "
      f"delta {delta:.5f}")
print("!! encoding is losing score - investigate before submitting" if delta > 0.002
      else ">>> encoding is lossless")
STATS["parity_delta"] = float(delta)
del vmaps
for k in RESULTS:
    RESULTS[k]["vmaps"] = None
gc.collect()

parity on 25 images | in-memory 0.8245 | after RLE 0.8245 | delta 0.00000
>>> encoding is lossless


0

In [18]:
# ── 16 · Test inference, submission.csv, final push ───────────────────────────
def build_test(names):
    acc = None
    for n in names:
        mp = load_maps(f"{CFG.RUN_TAG}_{n}", "test")
        if acc is None:
            acc = {k: mp[k].astype(np.float32) for k in mp}
        else:
            for k in acc:
                acc[k] += mp[k].astype(np.float32)
        del mp; gc.collect()
    for k in acc:
        acc[k] = np.clip(acc[k] / len(names), 0, 255).astype(np.uint8)
    return acc

ENS_TEST = build_test(BEST["names"])
info = write_submission(WORK / "submission.csv", ENS_TEST, TEST_IDS, POST)
shutil.copy2(WORK / "submission.csv", ART / "submission_stage8.csv")

CAND = [dict(name=BEST_LABEL, local_f1=BEST["f1"], **info)]
for m in MEMBER_RESULTS:
    CAND.append(dict(name=f"single_{m['member']}_defaultpost", local_f1=m["tta_f1"],
                     path=m["path"], rows=m["rows"], mean=m["mean"],
                     min=m["min"], max=m["max"], fallbacks=m["fallbacks"]))
cand = pd.DataFrame(CAND).sort_values("local_f1", ascending=False)
cand.to_csv(ART / "candidate_submissions.csv", index=False)
print(cand.to_string(index=False))
print(f"\nGT mean instances/image {STATS['gt_mean_instances']:.1f} | "
      f"stage 8 test predictions/image {info['mean']:.1f}")
STATS["submission"] = info

lines = [
    "# Stage 8 report - non-SSL ConvNeXt-UNet pair (Colab)",
    f"- run tag: {CFG.RUN_TAG} | encoder: {CFG.ENCODER} (plain ImageNet-21k, NO SimSiam)",
    f"- members: {[(m['member'], m['erosion']) for m in MEMBER_RESULTS]}",
    f"- fold {CFG.FOLD}/{CFG.N_FOLDS} | train/val {len(train_ids)}/{len(val_ids)} | "
    f"groups {STATS['n_groups']} | max val-to-train corr {STATS['max_val_train_corr']:.3f}",
    f"- split matches Stage 7: {(len(train_ids), len(val_ids), STATS['n_groups']) == (CFG.EXPECT_TRAIN, CFG.EXPECT_VAL, CFG.EXPECT_GROUPS)}",
    "- per-source local F1: " + ", ".join(f"{k} {v['f1']:.4f}" for k, v in RESULTS.items()),
    f"- best source: {BEST_LABEL} | post-processing {POST}",
    f"- RLE parity delta: {STATS['parity_delta']:.5f}",
    f"- test predictions/image: mean {info['mean']:.1f} "
    f"(min {info['min']}, max {info['max']}), fallbacks {info['fallbacks']}",
    f"- total elapsed: {fmt(elapsed())}",
    "",
    "## What Stage 9 does with this",
    "These maps are the NON-SSL family. Stage 9 (Kaggle, inside the Stage 7 notebook) blends",
    "them against m1/m2/m3 and sweeps the blend weight. Both families were scored on the same",
    "86 validation images - verify with val_ids.json before trusting any blend number.",
    "",
    "## Reading the local number",
    "Stages 2/4/5 fall on public = 0.773 * local - 0.111. Stage 7 (SSL) landed +0.016 above it,",
    "half of Stage 6's +0.033, so the SSL bonus is shrinking as local F1 rises. Treat this",
    "non-SSL model as sitting ON the line: local 0.82 projects to roughly 0.523.",
    "Public differences under ~0.008 are inside the noise of a 91-image split.",
]
(ART / "REPORT.md").write_text("\n".join(lines))
with open(ART / "stats.json", "w") as f:
    json.dump(STATS, f, indent=2, default=str)
print("\n".join(lines))

stage_for_push([ART / "REPORT.md", ART / "stats.json", ART / "candidate_submissions.csv",
                ART / "submission_stage8.csv", ART / "val_ids.json"]
               + [ART / f"sweep_{k}.csv" for k in RESULTS])
push_dataset("stage 8 - complete")

                 name  local_f1                                      path  rows       mean  min  max  fallbacks
            ens_n1_n2  0.807331              /content/work/submission.csv 20261 111.324176   12  639          0
single_n2_defaultpost  0.803205 /content/work/artifacts/submission_n2.csv 20158 110.758242   13  618          0
single_n1_defaultpost  0.788932 /content/work/artifacts/submission_n1.csv 20736 113.934066   14  631          0

GT mean instances/image 125.2 | stage 8 test predictions/image 111.3
# Stage 8 report - non-SSL ConvNeXt-UNet pair (Colab)
- run tag: stage8_nonssl | encoder: convnext_tiny.fb_in22k_ft_in1k (plain ImageNet-21k, NO SimSiam)
- members: [('n1', 0.45), ('n2', 0.58)]
- fold 0/5 | train/val 338/86 | groups 421 | max val-to-train corr 0.665
- split matches Stage 7: True
- per-source local F1: n1 0.7962, n2 0.8055, ens_n1_n2 0.8073
- best source: ens_n1_n2 | post-processing {'t_cell': 0.5, 't_seed': 0.75, 'min_area': 70, 'score_thr': 0.0}
- RLE parity d

In [19]:
# ── 17 · Download submission.csv to your device ───────────────────────────────
print("submission:", WORK / "submission.csv",
      f"({(WORK / 'submission.csv').stat().st_size / 1e6:.1f} MB)")
print("local F1", f"{BEST['f1']:.4f}", "| source", BEST_LABEL, "| post", POST)
print("\nAlso in the Kaggle dataset as submission_stage8.csv, in case the download below "
      "fails because the browser tab went to sleep.")
try:
    from google.colab import files
    files.download(str(WORK / "submission.csv"))
except Exception as exc:
    print("automatic download unavailable:", type(exc).__name__, exc)
    print("Grab it from the file browser at /content/work/submission.csv, or from the dataset.")
print("\ntotal elapsed:", fmt(elapsed()))

submission: /content/work/submission.csv (4.9 MB)
local F1 0.8073 | source ens_n1_n2 | post {'t_cell': 0.5, 't_seed': 0.75, 'min_area': 70, 'score_thr': 0.0}

Also in the Kaggle dataset as submission_stage8.csv, in case the download below fails because the browser tab went to sleep.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


total elapsed: 0h 55m 20s
